In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-3.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-9.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-5.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-6.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-7.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-2.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-4.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-1.txt
/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-8.txt
/kaggle/input/notebooks/aabdollahii/verbalizer/kg_sentences_all.txt
/kaggle/input/notebooks/aabdollahii/verbalizer/__results__.html
/kaggle/input/notebooks/aabdollahii/verbalizer/kg_sentences_removed_lines_sample.txt
/kaggle/input/notebooks/aabdollahii/verbalizer/__noteb

In [1]:
import os
import re
import random

import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

MODEL_NAME = "sbunlp/fabert"

FACTUAL_PATH = "/kaggle/input/notebooks/aabdollahii/verbalizer/kg_sentences_all_clean.txt"
GENERAL_PATH = "/kaggle/input/datasets/miladfa7/persian-wikipedia-dataset/Persian-WikiText-1.txt"

OUTPUT_DIR = "/kaggle/working/fabert_kg_mlm"

SEED = 42
MAX_LENGTH = 128
MLM_PROB = 0.15

FACTUAL_RATIO = 0.8
MIN_LEN_CHARS = 3

MAX_FACTUAL_LINES = 800_000
MAX_GENERAL_LINES = 200_000

random.seed(SEED)
torch.manual_seed(SEED)


def basic_cleanup(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\[https?://[^\]]+\]", "", text)
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text)
    text = re.sub(r"<ref[^/]*/>", "", text)
    text = text.strip()
    return text


def simple_normalize(text: str) -> str:
    text = basic_cleanup(text)
    text = text.replace("\u064A", "\u06CC")  # Arabic Yeh -> Farsi Yeh
    text = text.replace("\u0643", "\u06A9")  # Arabic Kaf -> Farsi Kaf
    text = text.strip()
    return text


def load_lines(path: str, limit: int = None):
    if path is None or not os.path.exists(path):
        return []
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            s = simple_normalize(line)
            if len(s) < MIN_LEN_CHARS:
                continue
            lines.append(s)
    return lines


factual_lines = load_lines(FACTUAL_PATH, limit=MAX_FACTUAL_LINES)
general_lines = load_lines(GENERAL_PATH, limit=MAX_GENERAL_LINES) if GENERAL_PATH else []

if GENERAL_PATH and len(general_lines) == 0:
    print("Warning: GENERAL_PATH is set but no general lines were loaded.")

if len(factual_lines) == 0:
    raise ValueError("No factual lines loaded. Check FACTUAL_PATH and preprocessing.")

if general_lines:
    n_total = len(factual_lines) + len(general_lines)
    n_factual_target = int(FACTUAL_RATIO * n_total)
    n_general_target = n_total - n_factual_target

    factual_sample = (
        factual_lines
        if len(factual_lines) <= n_factual_target
        else random.sample(factual_lines, n_factual_target)
    )
    general_sample = (
        general_lines
        if len(general_lines) <= n_general_target
        else random.sample(general_lines, n_general_target)
    )

    all_lines = factual_sample + general_sample
    random.shuffle(all_lines)
else:
    all_lines = factual_lines

print("Number of factual lines:", len(factual_lines))
print("Number of general lines:", len(general_lines))
print("Total training lines:", len(all_lines))

ds = Dataset.from_dict({"text": all_lines})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized = ds.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB,
)

model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,       
    gradient_accumulation_steps=2,       
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=500,
    dataloader_num_workers=2,
    lr_scheduler_type="linear",
    fp16=True,                           
    logging_steps=200,
    save_steps=2000,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


Number of factual lines: 800000
Number of general lines: 97726
Total training lines: 815906


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,16.253148


KeyboardInterrupt: 